In [4]:
!pip install wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 94.4 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Imports**

In [6]:
!pip install neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.5 MB/s eta 0:00:00


In [16]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 37.4 MB/s eta 0:00:00


In [17]:
import pandas as pd
import numpy as np
import torch
from lightning.pytorch.loggers import WandbLogger

import numpy as np
import torch
import wandb


import torch
import pandas as pd
from neuralforecast.models import NBEATS
from neuralforecast import NeuralForecast
from sklearn.base import BaseEstimator, TransformerMixin

import wandb
import json


**WandB**

In [9]:
import wandb
wandb.login()

True

**Reading Data**

In [10]:
train_df = pd.read_csv("/content/drive/MyDrive/WalmartPrices/NBeats/train_parquet.csv")
val_df = pd.read_csv("/content/drive/MyDrive/WalmartPrices/NBeats/val_parquet.csv")
X_train = train_df[["unique_id", "Date", "y", "IsHoliday"]].copy()
y_train = train_df["y"].copy()
X_val = val_df[["unique_id", "Date", "y", "IsHoliday"]].copy()
y_val = val_df["y"].copy()
X_train.head()

,unique_id,Date,y,IsHoliday
0,10_1,2010-02-05,40212.84,False
1,10_1,2010-02-12,67699.32,True
2,10_1,2010-02-19,49748.33,False
3,10_1,2010-02-26,33601.22,False
4,10_1,2010-03-05,36572.44,False


**WMAE LOSS**

In [11]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

**Dataset Prepare**

In [12]:
def prepare_df(df):
    df = df.copy()

    df["ds"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["unique_id", "ds"])

    return df.drop(columns=["Date"])

**Model**

In [34]:
def build_model(h, input_size, lr):

    model = NBEATS(
        input_size=input_size,
        h=h,
        max_steps=1500,
        val_check_steps=100,
        batch_size=64,
        learning_rate=lr,
        stack_types=['identity', 'trend', 'seasonality'],
        n_blocks=[1, 1, 1],
        random_seed=42,
        start_padding_enabled=True,
        logger = WandbLogger(experiment=wandb.run),
    )

    return NeuralForecast(
        models=[model],
        freq="W-FRI"
    )

In [32]:
def train_and_eval(train_df, val_df, config):

    wandb.init(project="walmart-sales-forecasting", config=config)

    train_df = prepare_df(X_train)
    print(train_df.head())
    val_df = prepare_df(X_val)
    print(val_df.head())
    nf = build_model(
        h=config["h"],
        input_size=config["input_size"],
        lr=config["lr"]
    )

    nf.fit(train_df)

    preds = nf.predict()

    # align predictions with validation
    merged = val_df.merge(
        preds,
        on=["unique_id", "ds"],
        how="left"
    )

    merged = merged.fillna(0)

    score = wmae(
        merged["y"].values,
        merged["NBEATS"].values,
        merged["IsHoliday"].values
    )

    wandb.log({"wmae": score})

    wandb.finish()

    return score

**Training Methods**

In [ ]:
configs = [
    {"lr": lr, "h": h, "input_size": inp}
    for lr in [0.001, 0.0005, 0.0001]
    for h in [35]
    for inp in [20, 26, 36, 52]
]

best_score = float("inf")
best_config = None

for config in configs:

    score = train_and_eval(train_df, val_df, config)

    print(config, score)

    if score < best_score:
        best_score = score
        best_config = config

print("BEST:", best_config, best_score)